# CS 3110/5110: Data Privacy
## In-Class Exercises, week of 9/1/2025

In [ ]:
# Load the data and libraries
import pandas as pd
import numpy as np

adult = pd.read_csv('https://github.com/jnear/cs211-data-privacy/raw/master/homework/adult_with_pii.csv')

## Question 1

Write code to group the `adult` dataset by a single column and count the number of members in each group.

In [18]:
def group_one_count(col):
    return adult.groupby(col).size()

In [19]:
group_one_count('Education')

Education
10th              933
11th             1175
12th              433
1st-4th           168
5th-6th           333
7th-8th           646
9th               514
Assoc-acdm       1067
Assoc-voc        1382
Bachelors        5355
Doctorate         413
HS-grad         10501
Masters          1723
Preschool          51
Prof-school       576
Some-college     7291
dtype: int64

In [20]:
s = group_one_count('Education')
assert s['10th'] == 933
assert s['9th'] == 514
assert s['Some-college'] == 7291

## Question 2

Write code to group the `adult` dataset by two columns and count the number of members in each group.

In [21]:
def group_two_count(col1, col2):
    return adult.groupby([col1,col2]).size()

In [22]:
group_two_count('Occupation', 'Education')

Occupation        Education   
Adm-clerical      10th             38
                  11th             67
                  12th             38
                  5th-6th           6
                  7th-8th          11
                                 ... 
Transport-moving  Doctorate         1
                  HS-grad         825
                  Masters          10
                  Prof-school       3
                  Some-college    283
Length: 201, dtype: int64

In [23]:
s = group_two_count('Occupation', 'Education')
assert s['Transport-moving']['Doctorate'] == 1
assert s['Adm-clerical']['10th'] == 38

## Question 3

Write code to perform a differencing attack to determine Brenn McNeely's age using only the `mean` aggregation function over large groups.

In [30]:
def get_brenns_age():
    mean_total = adult['Age'].mean()
    mean_without_Brenn = adult[adult['Name'] != 'Brenn McNeely']['Age'].mean()
    size = len(adult)
    return mean_total * size - mean_without_Brenn * (size - 1)

brenns_age = get_brenns_age()

In [31]:
assert brenns_age == 38.0

In [44]:
df = adult[['Education', 'Marital Status', 'Target']]
df.head(10)

,Education,Marital Status,Target
0,Bachelors,Never-married,<=50K
1,Bachelors,Married-civ-spouse,<=50K
2,HS-grad,Divorced,<=50K
3,11th,Married-civ-spouse,<=50K
4,Bachelors,Married-civ-spouse,<=50K
5,Masters,Married-civ-spouse,<=50K
6,9th,Married-spouse-absent,<=50K
7,HS-grad,Married-civ-spouse,>50K
8,Masters,Never-married,>50K
9,Bachelors,Married-civ-spouse,>50K


## Question 4

Does the dataset `df` satisfy $k$-Anonymity for $k=3$? Why not?

In [63]:
df.groupby(['Education', 'Marital Status', 'Target']).size().min()

1

Dataset `df` does not satisfy $k$-Anonymity for $k=3$ because it includes combinations of quasi identifiers that occure less than 3 times.

## Question 5

What columns should we designate as quasi-identifiers in the dataset `df`, and why?

I think Education and Marital status should be designated as quasi identifiers,because they are not directly indentifying the person but could be used in reidentification if combined with other sources

## Question 6

Imagine the column `Target` is *not* a quasi-identifier, and we generalize the dataset to achieve $k$-Anonymity for $k=2$ as follows:

- Replace each education level below "HS Grad" with `< HS` and others with `>= HS`
- Replace marital status with `Married` and `Not Married`
- Delete rows as required to achieve $k$-anonymity for $k=2$

For which rows is a homogeneity attack possible, and why?

Homogeneity attack can happen when in one group all rows have the same Target value, so even if there are two or more rows for k=2, when they all show the same Target then having the Education and Marital Status values will eb enough to know the Target.

## Question 7

Write code to generalize the `Zip` column in the adult dataset.

In [58]:
def generalize_zip(zip, l):
    return (zip // (10 ** l)) * (10 ** l)
    

In [59]:
# Test cases
assert generalize_zip(47401, 0) == 47401
assert generalize_zip(47401, 1) == 47400
assert generalize_zip(47401, 2) == 47400
assert generalize_zip(47401, 3) == 47000
assert generalize_zip(47401, 4) == 40000
assert generalize_zip(47401, 5) == 0